In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
print("✅ Libraries imported successfully")


✅ Libraries imported successfully


## 1. Load Preprocessed Data


In [3]:
# Load preprocessed data from n3
data_path = '../../data/preprocessed/preprocessed_data.csv'
df = pd.read_csv(data_path)

print(f"✅ Loaded data: {df.shape}")
print(f"\nTarget distribution:")
print(df['label'].value_counts().sort_index())
print(f"\nLabel 0 (Fake): {(df['label'] == 0).sum()} rows")
print(f"Label 1 (Real): {(df['label'] == 1).sum()} rows")


✅ Loaded data: (4736, 28)

Target distribution:
label
0    3929
1     807
Name: count, dtype: int64

Label 0 (Fake): 3929 rows
Label 1 (Real): 807 rows


## 2. Separate by Label & Shuffle


In [4]:
# Separate by label
df_fake = df[df['label'] == 0].reset_index(drop=True)  # Label 0
df_real = df[df['label'] == 1].reset_index(drop=True)  # Label 1

print("="*80)
print("SEPARATING BY LABEL")
print("="*80)
print(f"\nOriginal shape: {df.shape}")
print(f"Fake news (label=0): {df_fake.shape[0]} rows")
print(f"Real news (label=1): {df_real.shape[0]} rows")

# Shuffle both dataframes
df_fake = df_fake.sample(frac=1, random_state=42).reset_index(drop=True)
df_real = df_real.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✅ Both dataframes shuffled")


SEPARATING BY LABEL

Original shape: (4736, 28)
Fake news (label=0): 3929 rows
Real news (label=1): 807 rows

✅ Both dataframes shuffled


## 3. Split Each Label: 80/10/10


In [5]:
def stratified_split(df, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, random_state=42):
    """
    Split dataframe into train/val/test
    """
    # First split: 80% train, 20% temp (val+test)
    train, temp = train_test_split(df, test_size=(1-train_ratio), random_state=random_state)
    
    # Second split: split temp 50/50 into val and test
    val_size = val_ratio / (val_ratio + test_ratio)
    val, test = train_test_split(temp, test_size=test_ratio/(val_ratio+test_ratio), random_state=random_state)
    
    return train, val, test

print("="*80)
print("SPLITTING: 80% TRAIN / 10% VAL / 10% TEST")
print("="*80)

# Split fake news (label=0)
fake_train, fake_val, fake_test = stratified_split(df_fake)
print(f"\nFake news (label=0):")
print(f"  Train: {fake_train.shape[0]} rows")
print(f"  Val:   {fake_val.shape[0]} rows")
print(f"  Test:  {fake_test.shape[0]} rows")

# Split real news (label=1)
real_train, real_val, real_test = stratified_split(df_real)
print(f"\nReal news (label=1):")
print(f"  Train: {real_train.shape[0]} rows")
print(f"  Val:   {real_val.shape[0]} rows")
print(f"  Test:  {real_test.shape[0]} rows")


SPLITTING: 80% TRAIN / 10% VAL / 10% TEST

Fake news (label=0):
  Train: 3143 rows
  Val:   393 rows
  Test:  393 rows

Real news (label=1):
  Train: 645 rows
  Val:   81 rows
  Test:  81 rows


## 4. Combine & Verify Balance


In [6]:
# Combine fake and real for each split
X_train = pd.concat([fake_train, real_train], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
X_val = pd.concat([fake_val, real_val], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
X_test = pd.concat([fake_test, real_test], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print("="*80)
print("COMBINED DATASET")
print("="*80)
print(f"\nTrain set: {X_train.shape[0]} rows")
print(f"  Label 0 (Fake): {(X_train['label'] == 0).sum()}")
print(f"  Label 1 (Real): {(X_train['label'] == 1).sum()}")
print(f"  Ratio: {(X_train['label'] == 0).sum() / X_train.shape[0]:.2%} fake, {(X_train['label'] == 1).sum() / X_train.shape[0]:.2%} real")

print(f"\nValidation set: {X_val.shape[0]} rows")
print(f"  Label 0 (Fake): {(X_val['label'] == 0).sum()}")
print(f"  Label 1 (Real): {(X_val['label'] == 1).sum()}")
print(f"  Ratio: {(X_val['label'] == 0).sum() / X_val.shape[0]:.2%} fake, {(X_val['label'] == 1).sum() / X_val.shape[0]:.2%} real")

print(f"\nTest set: {X_test.shape[0]} rows")
print(f"  Label 0 (Fake): {(X_test['label'] == 0).sum()}")
print(f"  Label 1 (Real): {(X_test['label'] == 1).sum()}")
print(f"  Ratio: {(X_test['label'] == 0).sum() / X_test.shape[0]:.2%} fake, {(X_test['label'] == 1).sum() / X_test.shape[0]:.2%} real")

print(f"\n{'-'*80}")
print(f"TOTAL: {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]} rows")
print(f"Train: {X_train.shape[0] / (X_train.shape[0] + X_val.shape[0] + X_test.shape[0]):.1%}")
print(f"Val:   {X_val.shape[0] / (X_train.shape[0] + X_val.shape[0] + X_test.shape[0]):.1%}")
print(f"Test:  {X_test.shape[0] / (X_train.shape[0] + X_val.shape[0] + X_test.shape[0]):.1%}")


COMBINED DATASET

Train set: 3788 rows
  Label 0 (Fake): 3143
  Label 1 (Real): 645
  Ratio: 82.97% fake, 17.03% real

Validation set: 474 rows
  Label 0 (Fake): 393
  Label 1 (Real): 81
  Ratio: 82.91% fake, 17.09% real

Test set: 474 rows
  Label 0 (Fake): 393
  Label 1 (Real): 81
  Ratio: 82.91% fake, 17.09% real

--------------------------------------------------------------------------------
TOTAL: 4736 rows
Train: 80.0%
Val:   10.0%
Test:  10.0%


## 5. Save Split Datasets


In [7]:
import os

# Create directory if not exists
split_dir = '../../data/splited'
os.makedirs(split_dir, exist_ok=True)

# Save datasets
train_path = os.path.join(split_dir, 'train_set.csv')
val_path = os.path.join(split_dir, 'val_set.csv')
test_path = os.path.join(split_dir, 'test_set.csv')

X_train.to_csv(train_path, index=False)
X_val.to_csv(val_path, index=False)
X_test.to_csv(test_path, index=False)

print("="*80)
print("DATASETS SAVED")
print("="*80)
print(f"\n✅ train_set.csv: {X_train.shape[0]} rows × {X_train.shape[1]} columns")
print(f"✅ val_set.csv:   {X_val.shape[0]} rows × {X_val.shape[1]} columns")
print(f"✅ test_set.csv:  {X_test.shape[0]} rows × {X_test.shape[1]} columns")
print(f"\n📁 Location: {os.path.abspath(split_dir)}")
print(f"\n🚀 Ready for embedding phase!")


DATASETS SAVED

✅ train_set.csv: 3788 rows × 28 columns
✅ val_set.csv:   474 rows × 28 columns
✅ test_set.csv:  474 rows × 28 columns

📁 Location: d:\Vietnamese-Fake-News-Detection\data\splited

🚀 Ready for embedding phase!


## Process Overview

```
original_data (9500 rows)
├── LABEL=0 (FAKE): 5000 rows
│   ├── shuffle
│   ├── 80% train:     4000 rows
│   ├── 10% val:        500 rows
│   └── 10% test:       500 rows
│
└── LABEL=1 (REAL): 4500 rows
    ├── shuffle
    ├── 80% train:     3600 rows
    ├── 10% val:        450 rows
    └── 10% test:       450 rows

FINAL RESULT:
├── train_set.csv:   7600 rows (4000 fake + 3600 real) = 80%
├── val_set.csv:      950 rows (500 fake + 450 real)   = 10%
└── test_set.csv:     950 rows (500 fake + 450 real)   = 10%

✅ Balanced ratio maintained across all splits!
```


# Data Split - Stratified Train/Val/Test

Load preprocessed_data từ n3, tách label 0 & 1 riêng biệt, shuffle từng cái, chia 80/10/10, ghép lại giữ ratio balanced